# ESM-C 600M Embeddings (Local, Apple Silicon) from a Multi-FASTA File

This notebook reads protein sequences from a multi-FASTA file and computes per-protein
embeddings **locally on a Mac (Apple Silicon)** using a 600M-parameter ESM-C-equivalent
model, run on the Metal (MPS) backend.

**Why `Synthyra/ESMplusplus_large` instead of `biohub/ESMC-600M` directly:**
The official `biohub/ESMC-600M` checkpoint declares a very new `model_type` ("esmc") that
has not yet landed in a mainline `transformers` release, so loading it directly currently
raises `ValueError: ... model type esmc but Transformers does not recognize this
architecture` unless you install `transformers` from source (a moving target). `Synthyra/
ESMplusplus_large` ("ESM++") is a faithful reimplementation of the same 600M ESM-C model,
distributed as ordinary `transformers` custom code (`trust_remote_code=True`), reported to
match the original to ~1e-9 MSE in bfloat16. It works with any reasonably recent
`transformers` install and defaults to the `sdpa` attention backend, which runs fine on MPS
(flash-attention is CUDA-only and not needed here).

**Pipeline:**
1. Install dependencies
2. Load the ESM++ 600M model onto the MPS device (with CPU fallback)
3. Load sequences from the FASTA file
4. Run a forward pass per batch and mean-pool per-residue embeddings
5. Save embeddings + sequence IDs to CSV

## 1. Install dependencies

In [ ]:
# !pip install "transformers>=4.40" torch biopython pandas tqdm accelerate


In [1]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from Bio import SeqIO
from transformers import AutoModelForMaskedLM, AutoConfig


/Users/uqsmor23/miniconda3/envs/ECRefSeq_Oag_db/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuration

Set the path to your input FASTA file and the output CSV path.

In [8]:
# --- User settings ---
FASTA_PATH = "data/wzy_proteins.faa"          # path to your multi-FASTA file # wzy_proteins.faa, wzx_proteins.faa and all_prot_2.faa
OUTPUT_CSV = "esmc600m_embeddings_wzy.csv"  # where to save the embeddings

MODEL_NAME = "Synthyra/ESMplusplus_large"  # ESM-C 600M-equivalent, standard transformers-compatible

# ESM-C has a context window limit of 2048 tokens; truncate very long sequences if needed.
MAX_LENGTH = 2048

# Batch size for local inference. Start small on MPS (e.g. 1-4) and increase if memory allows.
BATCH_SIZE = 2


## 3. Set up the device (MPS with CPU fallback)

`PYTORCH_ENABLE_MPS_FALLBACK=1` tells PyTorch to silently run any operation not yet
implemented on MPS on the CPU instead of raising an error — useful since MPS op coverage
isn't 100% complete.

In [9]:
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon GPU via MPS backend")
else:
    device = torch.device("cpu")
    print("MPS not available - falling back to CPU (this will be slower)")


Using Apple Silicon GPU via MPS backend


## 4. Load the ESM-C 600M-equivalent model

`trust_remote_code=True` is required since this model ships its architecture as custom code
alongside the weights (this is normal and expected for this repo — you're running code
published by the model author, not arbitrary code from your FASTA file or elsewhere).

We explicitly set `attn_backend = "sdpa"`, which is also the default, since flash-attention
requires CUDA and isn't relevant on Apple Silicon.

In [10]:
config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
config.attn_backend = "sdpa"

model = AutoModelForMaskedLM.from_pretrained(
    MODEL_NAME,
    config=config,
    trust_remote_code=True,
).to(device).eval()

tokenizer = model.tokenizer

print(f"Loaded {MODEL_NAME} on {device}")


Loading weights: 100%|██████████| 368/368 [00:00<00:00, 8212.44it/s]


Loaded Synthyra/ESMplusplus_large on mps


## 5. Load sequences from the multi-FASTA file

Each record's `id` is kept as the identifier for the output CSV, so make sure your FASTA
headers are informative (e.g. `>locus_tag|organism|...`).

In [11]:
def load_fasta(path, max_length=None):
    records = []
    for rec in SeqIO.parse(path, "fasta"):
        seq = str(rec.seq).upper().replace("*", "")
        if max_length is not None and len(seq) > max_length:
            print(f"Warning: truncating {rec.id} from {len(seq)} to {max_length} aa")
            seq = seq[:max_length]
        records.append((rec.id, seq))
    return records

records = load_fasta(FASTA_PATH, max_length=MAX_LENGTH)
print(f"Loaded {len(records)} sequences from {FASTA_PATH}")
records[:3]


Loaded 242 sequences from data/wzy_proteins.faa


[('ecoh~~~wzy-O1~~~GU299791.1~~~',
  'MRIKYKDNGYILCFLLCFILSYLVLLKSDYFPADFLPYTEIYDGTYGEINNIEPAFLYLTRLFHYLNFPYIFFAMLVCALCLSWKIKYARKIIKDSYIYLFLYVYVSFYVFLHEMTQLRIAIAVTMCYVSVYYYFYKNCVKHALPWMVLAILFHYSALLLFMSLFIYSYRGLLIVIIGFVICMSFLNVYADTIALYLPNEKIVNYLYSISSSLDNRNDLAIFNLNNIIFLSIFILIFYLSRYIKLNDNEAKFIKYVQCSGILAFCIFFLASGVPVIAYRTAELLRIFYPMALVLILSHIKNNNMRYFIAVIIVILSGLMLFITLRAVSIVGQGL'),
 ('ecoh~~~wzy-O10~~~AB811599.1~~~',
  'MFLYLSIFICISILSFFPYQKYLLKAIMIFLLLMCAFRSEFVDRDYTNYITLIFESANNLFYPIEISFKVISFIAVNMLGGYYFVFIIFALISIYLKYNIIKKSSPFLLLSVLVYYSNTFLLQDMTQIRAGVAGAIVLYSVLQYAKNKNIMSFVMLIFLASTFHITTLLFFIVLAFDTNVVSRRYIFIYFLTLFGACCCYYAGVNFLTLLRYIPVSYVQYKFSDYISQSQGADFVPVNIFSVTQLLHLLIVILSFIFSRKKILSRDKVLVIKIYSLSPLAFIILSSMPVFALRVSELFAISEIILLPFLPLLFRQKQFAKLAIIGICFIMFYINIYHLEILNEYSAL'),
 ('ecoh~~~wzy-O100~~~AB812045.1~~~',
  'MAFIRFVNFLIIFFISIDMGGELGIRNVSFLLLVLLIAYYIVTCKKIKFRLPYVLLIILFLGYPVASLFIGMSNNFDADTGISQISIFVFAFLYFLKRIIEEDNNLPLMPIQYFNNAMLIISLFYVLMLISYQFFPSIFNSIALYIHDRGAGFAGVRTENGVGFANIYLKSSLFLIFTYCCFLKRNTIKAAIII

## 6. Generate embeddings

For each batch of sequences:
1. Tokenize with padding
2. Run a forward pass and take the final layer's hidden states
   (`output.last_hidden_state`)
3. Mean-pool over real (non-padding, non-special-token) residue positions using the
   attention mask, to get one fixed-length vector per protein

Mean pooling with the attention mask is important here since we're batching sequences of
different lengths and don't want padding tokens to distort the average.

In [12]:
def mean_pool_hidden_states(hidden_states, attention_mask):
    """Mean-pool per-residue embeddings, excluding padding and BOS/EOS special tokens.

    hidden_states: (batch, seq_len, hidden_dim)
    attention_mask: (batch, seq_len) - 1 for real tokens, 0 for padding
    """
    mask = attention_mask.clone()
    mask[:, 0] = 0  # BOS
    seq_lengths = attention_mask.sum(dim=1)  # includes BOS+EOS+residues
    for i, length in enumerate(seq_lengths):
        eos_pos = int(length.item()) - 1
        if eos_pos >= 0:
            mask[i, eos_pos] = 0  # EOS

    mask = mask.unsqueeze(-1).to(hidden_states.dtype)  # (batch, seq_len, 1)
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)
    return (summed / counts).cpu().float().numpy()


all_ids = []
all_embeddings = []
failed = {}

batches = [records[i:i + BATCH_SIZE] for i in range(0, len(records), BATCH_SIZE)]

for batch in tqdm(batches, desc="Embedding sequences (MPS)"):
    batch_ids = [r[0] for r in batch]
    batch_seqs = [r[1] for r in batch]

    try:
        inputs = tokenizer(
            batch_seqs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            output = model(**inputs)

        # Final layer hidden states: (batch, seq_len, hidden_dim)
        last_hidden = output.last_hidden_state
        pooled = mean_pool_hidden_states(last_hidden, inputs["attention_mask"])

        all_ids.extend(batch_ids)
        all_embeddings.extend(pooled)
    except Exception as e:
        for seq_id in batch_ids:
            failed[seq_id] = str(e)

print(f"Successfully embedded {len(all_ids)} / {len(records)} sequences")
if failed:
    print(f"Failed to embed {len(failed)} sequences. Examples:")
    for seq_id, err in list(failed.items())[:5]:
        print(f"  {seq_id}: {err}")


Embedding sequences (MPS): 100%|██████████| 121/121 [01:15<00:00,  1.60it/s]

Successfully embedded 242 / 242 sequences


## 7. Save embeddings to CSV

Output format: one row per sequence. First column is the sequence ID, remaining columns
are the embedding dimensions (named `dim_0`, `dim_1`, ...).

In [13]:
embedding_dim = all_embeddings[0].shape[0]
col_names = [f"dim_{i}" for i in range(embedding_dim)]

df = pd.DataFrame(all_embeddings, columns=col_names)
df.insert(0, "id", all_ids)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {df.shape[0]} embeddings of dimension {df.shape[1] - 1} to {OUTPUT_CSV}")
df.head()


Saved 242 embeddings of dimension 1152 to esmc600m_embeddings_wzy.csv


,id,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,...,dim_1142,dim_1143,dim_1144,dim_1145,dim_1146,dim_1147,dim_1148,dim_1149,dim_1150,dim_1151
0,ecoh~~~wzy-O1~~~GU299791.1~~~,0.003153,0.004490,-0.000383,0.005915,-0.000286,0.008157,0.001212,-0.006709,0.001658,...,-0.007829,0.001662,-0.003184,0.011714,0.011211,0.019278,-0.021915,-0.013471,-0.001359,-0.003808
1,ecoh~~~wzy-O10~~~AB811599.1~~~,0.006240,0.002716,-0.001595,0.006638,0.002829,0.005129,0.001711,-0.007976,0.001036,...,-0.004956,0.005167,-0.000601,0.015118,0.013246,0.016025,-0.016590,-0.015277,0.007605,-0.002302
2,ecoh~~~wzy-O100~~~AB812045.1~~~,0.003887,0.008312,0.007219,0.018973,0.003937,0.008778,-0.005456,-0.003045,0.000688,...,0.004136,-0.005228,0.005672,0.007681,0.003886,0.018041,-0.022318,-0.011467,0.002916,0.014145
3,ecoh~~~wzy-O102~~~JX087966.1~~~,-0.001162,0.005090,0.008947,0.008489,-0.000386,0.007168,-0.002696,-0.002362,-0.000987,...,-0.002383,-0.007255,0.008492,0.016317,0.008733,0.020749,-0.022149,-0.011955,-0.000562,0.000936
4,ecoh~~~wzy-O102~~~AB812047.1~~~,-0.000866,0.004956,0.009068,0.009927,0.000525,0.007486,-0.003037,-0.002566,-0.000782,...,-0.001872,-0.007352,0.008343,0.016687,0.008479,0.020460,-0.023782,-0.011438,-0.000359,0.003138


## Troubleshooting on Apple Silicon

- **"MPS not available"**: requires macOS 12.3+ and a PyTorch build with MPS support
  (`pip install torch` from a recent version should include it by default).
- **Unsupported operator errors**: these should be caught by `PYTORCH_ENABLE_MPS_FALLBACK=1`
  and silently run on CPU instead, but if you still see a hard failure, try lowering
  `BATCH_SIZE`.
- **Out-of-memory / system slowdown**: unified memory is shared with the rest of macOS, so
  a very large `BATCH_SIZE` can pressure the whole system, not just "VRAM" as on a discrete
  GPU. Lower `BATCH_SIZE` if you notice this.
- **`trust_remote_code=True` prompt**: the first time you load this model, `transformers`
  may ask you to confirm you trust the remote code. This is expected for this repo.
- **Wanting the "official" biohub/ESMC-600M checkpoint instead**: that will work once the
  `esmc` architecture lands in a released `transformers` version, or if you install
  `transformers` from source per the original error message. This notebook uses ESM++
  instead specifically to sidestep that dependency for now, at the cost of relying on a
  (well-validated) third-party reimplementation rather than the original checkpoint.

## Next steps

- Load this CSV alongside your labels (e.g. `wzx` / `wzy` / `other`) and train a probing
  classifier (logistic regression, linear SVM, or a small MLP) on the frozen embeddings.
- Cluster/deduplicate sequences (e.g. with mmseqs2 or cd-hit) *before* splitting into
  train/test to avoid leakage from near-duplicate homologs.
- Consider PCA/UMAP on this embedding matrix as a quick sanity check for class separability
  before training a classifier.